In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [4]:
from langchain_community.document_loaders import PyPDFLoader

c:\Users\athir\miniconda3\envs\qanda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [5]:
cd d:/Projects/Question-Answer-Creator-Application/

d:\Projects\Question-Answer-Creator-Application


In [ ]:
file_path = "data/statistics.pdf"
loader = PyPDFLoader(file_path)
data = loader.load()
data

[Document(metadata={'producer': 'Skia/PDF m120 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Report - Importance and the use of correlation in Statistics', 'source': 'data/stats.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='ImportanceandtheuseofcorrelationinStatistics\nIntroduction\nCorrelationisastatistical measurethat expressestheextent towhichtwovariablesarelinearlyrelated. It isacommontool for describingsimplerelationshipswithout makingastatement about causeandeffect.Correlationcoefficientsrangefrom-1to1, withavalueof 0indicatingnolinear relationshipbetweenthetwovariables, avalueof 1indicatingaperfect positivelinear relationship, andavalueof -1indicatingaperfectnegativelinear relationship.\nCorrelationisimportantinstatisticsbecauseitcanbeusedto\n1. Identifyrelationshipsbetweenvariables: Correlationcanbeusedtoidentifywhether thereisarelationshipbetweentwovariables, andif so, whether therelationshipispositiveornegative. Thisinformatio

In [7]:
content = ""
for page in data:
    content += page.page_content
content

'ImportanceandtheuseofcorrelationinStatistics\nIntroduction\nCorrelationisastatistical measurethat expressestheextent towhichtwovariablesarelinearlyrelated. It isacommontool for describingsimplerelationshipswithout makingastatement about causeandeffect.Correlationcoefficientsrangefrom-1to1, withavalueof 0indicatingnolinear relationshipbetweenthetwovariables, avalueof 1indicatingaperfect positivelinear relationship, andavalueof -1indicatingaperfectnegativelinear relationship.\nCorrelationisimportantinstatisticsbecauseitcanbeusedto\n1. Identifyrelationshipsbetweenvariables: Correlationcanbeusedtoidentifywhether thereisarelationshipbetweentwovariables, andif so, whether therelationshipispositiveornegative. Thisinformationcanbeuseful for understandingtherelationshipsbetweendifferent factorsinacomplexsystem.\n2. Makepredictions: If thereisastrongcorrelationbetweentwovariables, it ispossibletousethevalueof onevariabletopredictthevalueof theother variable. Thiscanbeuseful for makingprediction

In [8]:
from langchain_text_splitters import TokenTextSplitter
from transformers import AutoTokenizer

# Load a specific Hugging Face tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [9]:
# Create the splitter using the loaded tokenizer's encode method
text_splitter = TokenTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=200,
    chunk_overlap=50
)

In [10]:
# Split your content
chunks = text_splitter.split_text(content)

In [11]:
print(len(content),len(chunks))

3664 6


In [13]:
from langchain_core.documents import Document

# convert string chunks to Document format
document_chunks = [Document(page_content=chunk) for chunk in chunks]

In [14]:
document = Document(page_content=content)

In [15]:
from langchain_groq import ChatGroq

llamaChatModel = ChatGroq(model="llama-3.3-70b-versatile",temperature = 0.3)
openaiChatModel =  ChatGroq(model="openai/gpt-oss-120b",temperature = 0.3)
qwenChatModel =  ChatGroq(model="qwen/qwen3-32b",temperature = 0.3)

For question generation, we directly pass content to model. For Aswer generation, we store the content in vector database and do retrieval.

In [16]:
question_prompt_template = """
You are an expert at creating questions based on given materials and documentation.
Your goal is to prepare a student for their exam.
You do this by asking questions about the text below:

------------
{text}
------------

Create questions that will prepare the students for their exams.
Make sure not to loose any important information.
Respond with only questions. No other information.

QUESTIONS:
"""

In [17]:
from langchain_core.prompts import PromptTemplate

question_prompt = PromptTemplate(template = question_prompt_template, input_variables = ["text"])

In [18]:
refine_question_template = """
You are an expert at creating questions based on materials and documentation.
Your goal is to help a student to prepare for their exam.
We have recieved some questions to a certain extent : 
{existing_answer}

We have the option to refine the existing questions or add the new ones (only if necessary) with some more context below.

------------
{text}
------------

Given the new context, refine the original questions in English.
If the context is not helpful, please provided the original question.
Respond with only questions. No other information.

QUESTIONS:
"""

In [19]:
refine_question_prompt = PromptTemplate(
    input_variables = ["existing_answer","text"],
    template = refine_question_template
)

## Legacy chain

In [ ]:
from langchain_classic.chains.summarize import load_summarize_chain

question_chain = load_summarize_chain(
    llm = openaiChatModel,
    chain_type = "refine",
    verbose = True,
    question_prompt = question_prompt,
    refine_prompt = refine_question_prompt
)

questions = question_chain.run([document])
print(questions)

## LECL

In [22]:
from langchain_core.output_parsers import StrOutputParser

# Define the chains using LCEL
initial_chain = question_prompt | openaiChatModel | StrOutputParser()

refine_chain = refine_question_prompt | openaiChatModel | StrOutputParser()

print("Generating questions using LCEL pattern...")

# Start with the first chunk
current_questions = initial_chain.invoke({"text": document_chunks[0].page_content})

# Iteratively refine with the rest of the chunks
for i, doc in enumerate(document_chunks[1:], start=2):
    print(f"Refining with chunk {i}/{len(document_chunks)}...")
    current_questions = refine_chain.invoke({
        "existing_answer": current_questions,
        "text": doc.page_content
    })

print("\nFinal Questions Generated!")
print(current_questions)

Generating questions using LCEL pattern...
Refining with chunk 2/6...
Refining with chunk 3/6...
Refining with chunk 4/6...
Refining with chunk 5/6...
Refining with chunk 6/6...

Final Questions Generated!
1. What does the correlation coefficient quantify in a data set, and how should its numeric value be interpreted to describe the strength (weak, moderate, strong) and direction (positive or negative) of a linear association between two variables?  

2. Even when a correlation coefficient is large (e.g., close to ±1), why does it not constitute evidence of a causal relationship, and how can a third variable (confounder) such as weather create a spurious correlation like ice‑cream sales and shark attacks?  

3. What are the theoretical limits of a correlation coefficient (‑1 and +1), and what does each extreme value tell us about the exact linear relationship between the two variables?  

4. What does a correlation coefficient of 0 imply about the existence of a linear relationship bet

In [1]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\athir\miniconda3\envs\qanda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
